#📦 Basic Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from wordcloud import WordCloud
import nltk
from nltk.corpus import stopwords

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

from nltk.stem.porter import PorterStemmer
import string
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
# Read the CSV file
df = pd.read_csv(r'C:\Users\ASUS\Desktop\MLOPS\ML-Pipeline-DVC-AWS-S3\Data\combined_data.csv')



#🧹 Data Preprocessing

In [4]:
df.tail(10)

,label,text
83438,1,downloadable software ds is a fast paced compa...
83439,1,http printlost hk viagra escapenumber pills x ...
83440,0,howstuffworks r lifestyle april escapenumber e...
83441,1,lowest prices hbie qsxh gycj swlw swsszguohc z...
83442,0,tewk wrote patch was to large to attach so htt...
83443,0,hi given a date how do i get the last date of ...
83444,1,now you can order software on cd or download i...
83445,1,dear valued member canadianpharmacy provides a...
83446,0,subscribe change profile contact us long term ...
83447,1,get the most out of life ! viagra has helped m...


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 83448 entries, 0 to 83447
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   label   83448 non-null  int64 
 1   text    83448 non-null  object
dtypes: int64(1), object(1)
memory usage: 1.3+ MB


In [6]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
df['label'] = encoder.fit_transform(df['label'])

# Remove duplicates
df = df.drop_duplicates(keep='first')


#🛠 Feature Engineering

In [7]:


ps = PorterStemmer()
stop_words = set(stopwords.words('english'))

def transform_text(text):
    # Lowercase
    text = text.lower()
    # Tokenize
    tokens = nltk.word_tokenize(text)
    # Keep only alphanumeric
    tokens = [t for t in tokens if t.isalnum()]
    # Remove stopwords
    tokens = [t for t in tokens if t not in stop_words]
    # Stem
    tokens = [ps.stem(t) for t in tokens]
    return " ".join(tokens)

df['transformed_text'] = df['text'].apply(transform_text)


#🔢 Vectorization

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfid = TfidfVectorizer(max_features=500)
X = tfid.fit_transform(df['transformed_text']).toarray()
y = df['label'].values



#✂️ Train-Test Split

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=2)


#🤖 Model Training

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, BaggingClassifier, ExtraTreesClassifier, GradientBoostingClassifier


svc = SVC(kernel="sigmoid", gamma=1.0)
knc = KNeighborsClassifier()
mnb = MultinomialNB()
dtc = DecisionTreeClassifier(max_depth=5)
lrc = LogisticRegression(solver='liblinear', penalty='l1')
rfc = RandomForestClassifier(n_estimators=50, random_state=2)
abc = AdaBoostClassifier(n_estimators=50, random_state=2)
bc = BaggingClassifier(n_estimators=50, random_state=2)
etc = ExtraTreesClassifier(n_estimators=50, random_state=2)
gbdt = GradientBoostingClassifier(n_estimators=50, random_state=2)


clfs = {
    'SVC': svc,
    'KNN': knc,
    'NB': mnb,
    'DT': dtc,
    'LR': lrc,
    'RF': rfc,
    'Adaboost': abc,
    'Bgc': bc,
    'ETC': etc,
    'GBDT': gbdt
}


#📊 Model Evaluation

In [11]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


def train_classifier(clfs, X_train, y_train, X_test, y_test):
    clfs.fit(X_train, y_train)
    y_pred = clfs.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    return accuracy, precision, recall, f1

results = []

for name, clf in clfs.items():
    acc, prec, rec, f1 = train_classifier(clf, X_train, y_train, X_test, y_test)
    results.append([name, acc, prec, rec, f1])
    print(f"\nFor: {name}")
    print("Accuracy:", acc)
    print("Precision:", prec)
    print("Recall:", rec)
    print("F1 Score:", f1)

import pandas as pd
results_df = pd.DataFrame(results, columns=['Model', 'Accuracy', 'Precision', 'Recall', 'F1'])
print("\nSummary:\n", results_df.sort_values(by='F1', ascending=False))



For: SVC
Accuracy: 0.8632714200119832
Precision: 0.8680036256514843
Recall: 0.8727500569605833
F1 Score: 0.8703703703703703


c:\Users\ASUS\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\ASUS\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "c:\Users\ASUS\anaconda3\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\ASUS\anaconda3\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^


For: KNN
Accuracy: 0.9147393648891552
Precision: 0.8744527033906934
Recall: 0.9783549783549783
F1 Score: 0.9234905102424862

For: NB
Accuracy: 0.9216896345116836
Precision: 0.9082067533602884
Recall: 0.9467988152198679
F1 Score: 0.9271013441909756

For: DT
Accuracy: 0.854823247453565
Precision: 0.7957189390414147
Recall: 0.974025974025974
F1 Score: 0.8758899759258311

For: LR
Accuracy: 0.9575194727381666
Precision: 0.9481284016438964
Recall: 0.9724310776942355
F1 Score: 0.9601259771666385

For: RF
Accuracy: 0.9710605152786099
Precision: 0.9653315382026254
Recall: 0.9801777170198223
F1 Score: 0.9726979820247583

For: Adaboost
Accuracy: 0.8658478130617137
Precision: 0.8091141155337052
Recall: 0.974937343358396
F1 Score: 0.8843192973391888

For: Bgc
Accuracy: 0.9628520071899341
Precision: 0.9578002244668912
Recall: 0.9722032353611301
F1 Score: 0.964947987336047

For: ETC
Accuracy: 0.9748352306770521
Precision: 0.9725237449118046
Recall: 0.9798359535201641
F1 Score: 0.9761661559414369

Fo